When we implemented our bottleneck algorithm in C (or another compiled language), but is still too slow, then we should focus on the computations and memory access patterns in more detail.


## Rewriting Calculations

When a bottleneck algorithm is CPU-bound, i.e. when the calculations performed on the CPU are the time-limiting factor, we could look into rewriting these calculations. When the same calculation is repeated for millions of pixel values, even a small reduction in *computational cost* per pixel could lead to a substantial improvement in speed. Useful techniques to reduce computational cost are *reformulating equations*, *avoiding repeated operations*, *lookup-tables*, and *approximation*.

Note that the effect has to be significant in order to outweigh the downside of (possible) less readable and maintainable code. Maintaining (someone else's) ugly code can be a bottleneck too!


### Reformulating Equations

Try to keep in mind that addition, subtraction and multiplication are cheaper than division and taking square roots. And that exponentials, floating point powers, and logarithms are even more expensive. Can you think of ways to remove the most expensive operations from your equations? Or perhaps there are ways to just reduce the number of operations?

Suppose we would like to check if the Euclidean distance $d$ between $(x_1, y_1, z_1)$ and $(x_2, y_2, z_2)$ is smaller than some value $r$. We could implement the calculation of the distance in C as:

```c
const double d = sqrt(pow(x1 - x2, 2.0) + pow(y1 - y2, 2.0) + pow(z1 - z2, 2.0));

if (d < r) {
    ...
```

But a more efficient version is:

```c
const double dx = x1 - x2;
const double dy = y1 - y2;
const double dz = z1 - z2;

const double d = sqrt(dx*dx + dy*dy + dz*dz);

if (d < r) {
    ...
```

This avoids the general-purpose `pow()` function, replacing it with multiplication. Since each exponent is exactly 2, multiplication is simpler and cheaper.

And... do we actually need to square root at all? The purpose was to check if $d$ is smaller than $r$. This is equivalent to checking if $d^2$ is smaller than $r^2$! Bye bye square root:

```c
const double dx = x1 - x2;
const double dy = y1 - y2;
const double dz = z1 - z2;

const double d2 = dx*dx + dy*dy + dz*dz;

if (d2 < r*r) {
    ...
```

Other popular candidates for simplification are polynomials. You might be tempted to write $y = a_0 + a_1 x + a_2 x^2 + a_3 x^3$ as:

```c
y[i] = a0 + a1 * x[i] + a2 * x[i] * x[i] + a3 * x[i] * x[i] * x[i];
```

But calculations involving polynomials can be rewritten into a nested form, like this:

```c
y[i] = a0 + x[i] * (a1 + x[i] * (a2 + x[i] * a3));
```

This not only reduces the number of multiplications, but it also allows the compiler to instruct the CPU to perform three *fused multiply-add* (FMA) instructions. An FMA calculates $a \cdot b + c$ in one operation.


### Avoiding Repeated Operations

In for-loops you may encounter equations that are (partly) dependent on the loop variable, while other equations are not. Take, for example, this for loop:

```c
for (int i = 1; i < n - 1; i++) {
    sum = sum + h * (data[i - 1] + 4.0 * data[i] + data[i + 1]) / 3.0;
}
```

The part `(data[i - 1] + 4.0 * data[i] + data[i + 1])` depends on the loop variable `i`, but the part `h / 3.0` does not. Taking part that out of the loop saves a multiplication and an expensive division:

```c
for (int i = 1; i < n - 1; i++) {
    sum = sum + data[i - 1] + 4.0 * data[i] + data[i + 1];
}

sum *= h / 3.0;
```


### Lookup-Tables

Now you know that operations such as division and taking square roots are expensive, you can image that it makes sense to e.g. pre-calculate an inverse value and multiply with that value in a for loop:

```c
const double sigma_inv = 1.0 / sigma;
for (int i = 0; i < n; i++) {
    for (int j = 0; i < n; i++) {
        const double d = (data[i] - data[j]) * sigma_inv;
    ...
```

However, instead of just pre-calculating single values we could also pre-calculate entire convolution kernels. The example below shows how to allocate a small array and pre-calculate a Gaussian kernel. This allows to simply look up the value `gauss[j + k]` in our for-loop, instead of calculating `exp(-(j * j) / (2.0 * sigma * sigma))` over and over again. The array `gauss` is called a lookup-table:

```c
// Allocate a variable-length array for the lookup-table.
double gauss[2 * k + 1];

// Pre-compute the lookup table values.
double w = 0.0;
for (int j = -k; j <= k; j++) {
    gauss[j + k] = exp(-(j * j) / (2.0 * sigma * sigma));
    w += gauss[j + k];
}
for (int j = -k; j <= k; j++) {
    gauss[j + k] /= w;
}

// Convolve data with the Gaussian kernel.
for (int i = k; i < n - k; i++) {
    result[i] = 0.0;
    for (int j = -k; j <= k; j++) {
        result[i] += gauss[j + k] * data[i + j];
    }
}
```


### Approximations

How precise should your calculations be? Many expensive operations can be replaced by a less precise, but cheaper alternative. Such approximations may use techniques like interpolating in pre-calculated lookup-tables, iterative methods, polynomial approximations, or combinations thereof. There numerous approximations to all kinds of math functions published online.
